# Reema — Coordinator agent testing outputs

AAI-520 Final Team Project: Multi-Agent Financial Analysis System.

Tests for coordinator planning, routing, retry behavior, memory, error handling, and input validation. **Deterministic mock tests:** Ollama and live Yahoo Finance are not used; run `integration_demo.py` for a real end-to-end financial analysis.

In [1]:
!git clone https://github.com/roui2025/-Agentic-Financial-Analysis-System-AAI520.git

import os
os.chdir("/content/-Agentic-Financial-Analysis-System-AAI520")

Cloning into '-Agentic-Financial-Analysis-System-AAI520'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 70 (delta 29), reused 12 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 98.90 KiB | 3.41 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [2]:
import json
import tempfile
from pathlib import Path
from unittest.mock import patch
from coordinator_agent import plan_research, run_research
from agent_memory import AgentMemory

def model_reply(prompt):
    if "Review whether" in prompt:
        return json.dumps({"complete": True, "limitations": [], "next_steps": []})
    return json.dumps({"steps": ["Retrieve the requested evidence"], "routes": ["financial", "news"], "reason": "Evidence requested"})

print("Coordinator imports and test setup: PASS")

Coordinator imports and test setup: PASS


## Routing tests
Questions about finances, news, or both should route to the appropriate specialists.

In [3]:
questions = {
    "Revenue and earnings": ["financial"],
    "Latest headlines and sentiment": ["news"],
    "Financial performance and news": ["financial", "news"],
}
with patch("coordinator_agent.ask_llm", side_effect=model_reply):
    for question, expected in questions.items():
        actual = plan_research("AAPL", question)["routes"]
        assert actual == expected, (question, actual)
        print(f"PASS | {question}: {actual}")

PASS | Revenue and earnings: ['financial']
PASS | Latest headlines and sentiment: ['news']
PASS | Financial performance and news: ['financial', 'news']


## Invalid tool response retry
The coordinator should retry when the LLM invents a tool name.

In [4]:
bad = json.dumps({"steps": ["Look up"], "routes": ["get_stock_price"], "reason": "bad route"})
good = json.dumps({"steps": ["Look up"], "routes": ["financial"], "reason": "valid route"})
with patch("coordinator_agent.ask_llm", side_effect=[bad, good]) as mocked:
    plan = plan_research("MSFT", "Analyze stock price")
    assert plan["routes"] == ["financial"]
    assert mocked.call_count == 2
print("PASS | Invalid route retried; accepted:", plan["routes"])

PASS | Invalid route retried; accepted: ['financial']


## Agent dispatch, process reflection, and persistent memory
Mock specialists allow repeatable coordinator tests without network calls.

In [5]:
with tempfile.TemporaryDirectory() as folder:
    memory = AgentMemory(path=Path(folder) / "memory.json", max_notes=3)
    with patch("coordinator_agent.ask_llm", side_effect=model_reply):
        report = run_research(
            " aapl ", "Financial performance and news",
            news_agent=lambda ticker: {"summary": f"Headlines checked for {ticker}"},
            financial_agent=lambda ticker: {"summary": f"Financial metrics checked for {ticker}"},
            memory=memory,
        )
    assert report["ticker"] == "AAPL"
    assert list(report["results"]) == ["financial", "news"]
    assert report["reflection"]["complete"] is True
    assert len(memory.get("aapl")) == 1
    print("PASS | Normalized ticker:", report["ticker"])
    print("PASS | Specialist routes:", list(report["results"]))
    print("PASS | Process reflection:", report["reflection"]["complete"])
    print("PASS | Persistent notes:", len(memory.get("AAPL")))

PASS | Normalized ticker: AAPL
PASS | Specialist routes: ['financial', 'news']
PASS | Process reflection: True
PASS | Persistent notes: 1


## Failure handling and ticker validation
A specialist error should be captured in the report; malformed tickers should be rejected.

In [6]:
def broken_financial(ticker):
    raise RuntimeError("test-only data service outage")
with patch("coordinator_agent.ask_llm", side_effect=model_reply):
    failure_report = run_research(
        "AAPL", "Stock price",
        news_agent=lambda ticker: {"summary": "not used"},
        financial_agent=broken_financial,
    )
assert "test-only data service outage" in failure_report["results"]["financial"]["error"]
print("PASS | Specialist error captured:", failure_report["results"]["financial"]["error"])
try:
    run_research("BAD TICKER!", "Stock price", lambda t: {}, lambda t: {})
except ValueError as exc:
    print("PASS | Invalid ticker rejected:", str(exc))
else:
    raise AssertionError("Invalid ticker accepted")

PASS | Specialist error captured: test-only data service outage
PASS | Invalid ticker rejected: Enter a valid stock ticker.


## Results and limitations
All 8 checks above passed in a local Python test run against a copy of the current coordinator and memory modules. No network calls or actual model generations were made. For live, end-to-end demonstration, run the repository's `integration_demo.py` with Ollama and the project requirements installed.